[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/09_backtracking.ipynb)

*DSA Interview Handbook · Notebook 9 of 14*

# Recursion & Backtracking

Build answers one choice at a time, undo the choice, try the next — with pruning to skip dead branches.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Backtracking explores a maze: try a path, hit a dead end, step back, undo, try the next.

**What interviewers really test here:** The choose-explore-undo template, pruning and complexity.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Recursion: a function that calls itself *(Basics)*
- [ ] 2. The decision tree *(Basics)*
- [ ] 3. The backtracking template: choose, explore, un-choose *(Core)*
- [ ] 4. Permutations *(Core)*
- [ ] 5. Pruning and duplicates *(Core)*
- [ ] 6. Constraint problems *(Advanced)*
- [ ] 7. From backtracking to memoisation *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definitions

**Recursion** is a technique where a function solves a problem by calling itself on smaller instances of the same problem, until reaching a **base case** that is solved directly.



**Backtracking** is an algorithmic paradigm that builds a solution incrementally, one choice at a time, and **abandons (backtracks from)** a partial solution as soon as it determines it cannot lead to a valid complete solution. It is a depth-first search over the **state-space tree** of choices.

### Parts of a recursive function

1. **Base case** — the stopping condition; returns without recursing.
2. **Recursive case** — breaks the problem into a smaller sub-problem and calls itself.
3. **Progress** — each call must move toward the base case, or recursion never ends.



Each call creates a **stack frame** holding its parameters and local variables. Too many nested calls exhaust the stack: `RecursionError` in Python (default limit ~1000), stack overflow in other languages.

### Types of recursion

| Type | Description | Example |
| --- | --- | --- |
| **Direct** | function calls itself | `fact(n) = n * fact(n − 1)` |
| **Indirect** | A calls B, B calls A | `is_even ↔ is_odd` |
| **Tail** | the recursive call is the last action | `gcd(a, b) = gcd(b, a % b)` |
| **Non-tail (head)** | work remains after the call returns | `n * fact(n − 1)` |
| **Linear** | one recursive call per invocation | sum of a list |
| **Tree (multiple)** | two or more calls per invocation | Fibonacci, subsets, permutations |



Tail recursion can be turned into a loop (some languages do this automatically; Python does not).

### Analysing recursive algorithms

Write a **recurrence relation** for the running time, then solve it.



| Recurrence | Example | Solution |
| --- | --- | --- |
| T(n) = T(n − 1) + O(1) | factorial, linear recursion | O(n) |
| T(n) = T(n / 2) + O(1) | binary search | O(log n) |
| T(n) = 2T(n / 2) + O(n) | merge sort | O(n log n) |
| T(n) = 2T(n − 1) + O(1) | subsets, Tower of Hanoi | O(2ⁿ) |
| T(n) = T(n − 1) + T(n − 2) | naive Fibonacci | O(φⁿ) ≈ O(1.618ⁿ) |
| T(n) = n · T(n − 1) | permutations | O(n!) |



**Master theorem** for T(n) = a·T(n/b) + O(nᵈ): if d &gt; log\_b a → O(nᵈ); if d = log\_b a → O(nᵈ log n); if d &lt; log\_b a → O(n^(log\_b a)).



Space = maximum recursion depth × space per frame (plus any stored results).

### Backtracking: state-space tree and pruning

The **state-space tree** has the empty partial solution at the root; each edge is one choice; each leaf is a complete candidate. Backtracking explores it depth-first:




1. **Choose** an option and add it to the partial solution.
2. **Check constraints**; if violated, **prune** (don't go deeper).
3. **Explore** recursively.
4. **Un-choose** (undo) and try the next option.



Pruning with a **bounding function** (a quick test that a branch can't succeed) is what separates backtracking from brute-force enumeration. Its worst case is still exponential, but practical performance improves dramatically.



| Problem class | Search space |
| --- | --- |
| Subsets / combinations | 2ⁿ |
| k-combinations | C(n, k) |
| Permutations | n! |
| N-Queens | ≈ n! (one queen per row and column) |
| Sudoku | 9^(empty cells) |

### Recursion vs iteration, and related paradigms

|   | Recursion | Iteration |
| --- | --- | --- |
| Code | shorter, mirrors the definition | often longer |
| Memory | O(depth) stack frames | O(1) usually |
| Speed | function-call overhead | faster |
| Risk | stack overflow | infinite loops |
| Best for | trees, divide and conquer, backtracking | simple linear processing |



| Paradigm | Idea |
| --- | --- |
| Brute force | generate every candidate, test each |
| **Backtracking** | build candidates step by step, prune invalid partial ones |
| Branch and bound | backtracking for optimisation, pruning by a bound on the best possible value |
| Divide and conquer | split into *independent* sub-problems, combine (merge sort) |
| Dynamic programming | split into *overlapping* sub-problems, store results |

### Applications

- Puzzles: N-Queens, Sudoku, crosswords, mazes.
- Combinatorics: generating subsets, permutations, combinations, partitions.
- Constraint satisfaction: scheduling, timetabling, graph colouring.
- Parsing and regular-expression matching engines.
- Game trees (with minimax and alpha-beta pruning) in chess engines.
- Recursion generally: tree/graph traversal, divide-and-conquer sorts, fractals.

---
# Part 2: Lessons, step by step

## Lesson 1: Recursion: a function that calls itself

*Level: Basics*

A **recursive** function solves a problem by calling itself on a **smaller** version of the same problem. It needs two parts:




1. **Base case** — a version so small you answer it directly (and stop).
2. **Recursive case** — shrink the problem, call yourself, use the result.



```python
def factorial(n):
    if n <= 1:                  # base case
        return 1
    return n * factorial(n - 1) # smaller problem
```



Each call waits on the **call stack** until the smaller call returns. Forgetting the base case (or not shrinking) means infinite recursion → `RecursionError`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def call_stack_lesson():
    frames = []

    def fact(n):
        frames.append(f"fact({n})")
        if n <= 1:
            result = 1
        else:
            result = n * fact(n - 1)
        frames.pop()
        return result

    return fact(4)


call_stack_lesson()

24

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Call fact(4) — pushed onto the call stack
2. Call fact(3) — pushed onto the call stack
3. Call fact(2) — pushed onto the call stack
4. Call fact(1) — pushed onto the call stack
5. fact(1) returns 1 — popped
6. fact(2) returns 2 — popped
7. fact(3) returns 6 — popped
8. fact(4) returns 24 — popped

</details>


**✏️ Your turn:** you now have what you need for [Q14](#q14). Try one now, or carry on and do them in Part 3.

## Lesson 2: The decision tree

*Level: Basics*

Many problems are "make a sequence of choices": include this element or not, which number goes next, where to place the next queen. Draw every possible sequence of choices and you get a **decision tree**: each level is one choice, each leaf is one complete candidate.



```text
              []
         /          \          include 1?
       [1]           []
      /   \        /    \      include 2?
  [1,2]  [1]     [2]    []     ← 4 leaves = 4 subsets of [1, 2]
```



The tree can be huge: 2ⁿ leaves for subsets, n! for permutations. That's why these problems have exponential cost — and why **pruning** (cutting branches that can't work) is the key skill.

## Lesson 3: The backtracking template: choose, explore, un-choose

*Level: Core*

**Backtracking** walks the decision tree with DFS, keeping one shared `path`:



```python
def backtrack(start, path):
    if done(path):
        res.append(path[:])        # save a COPY
        return
    for i in range(start, len(choices)):
        if not ok(choices[i]):     # prune
            continue
        path.append(choices[i])    # choose
        backtrack(i + 1, path)     # explore
        path.pop()                 # un-choose (backtrack)
```



Using `start` and calling with `i + 1` means every combination is produced once, in increasing index order. For **subsets**, every node of the tree is an answer, so record the path at the start of each call.



**⚠️ Watch out:** `res.append(path)` without copying stores the same list again and again — at the end they're all empty.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q7](#q7), [Q8](#q8), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 4: Permutations

*Level: Core*

For permutations the order matters and every element is used, so each level may choose **any** unused element, not just later ones. Track which are used:



```python
def go():
    if len(path) == n:
        res.append(path[:]); return
    for i in range(n):
        if used[i]: continue
        used[i] = True; path.append(nums[i])
        go()
        path.pop(); used[i] = False
```



There are n! permutations, each of length n → O(n · n!).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def permute_lesson():
    nums = [1, 2, 3]
    path, used, res = [], [False] * 3, []

    def go():
        if len(path) == len(nums):
            res.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            go()
            path.pop()
            used[i] = False

    go()
    return res


permute_lesson()

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Complete permutation #1: [1, 2, 3]
2. Complete permutation #2: [1, 3, 2]
3. Complete permutation #3: [2, 1, 3]
4. Complete permutation #4: [2, 3, 1]
5. Complete permutation #5: [3, 1, 2]
6. Complete permutation #6: [3, 2, 1]

</details>


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q4](#q4). Try one now, or carry on and do them in Part 3.

## Lesson 5: Pruning and duplicates

*Level: Core*

**Pruning** = stop exploring a branch as soon as you know it can't lead to an answer.




- **Combination sum**: sort the candidates; once a candidate is bigger than the remaining target, every later one is too — `break`.
- **Count limits**: if fewer elements remain than you still need, stop.
- **Reuse allowed**: recurse with `i` instead of `i + 1`.



#### Avoiding duplicate answers



If the input has repeated values, sort it and skip a value equal to the previous one **at the same level**:



```python
for i in range(start, n):
    if i > start and nums[i] == nums[i - 1]:
        continue
```



The first copy explores every option; a second copy at the same level would only repeat it.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def pruning_lesson():
    cands, target = [2, 3, 5], 8
    res, path = [], []

    def go(start, remain):
        if remain == 0:
            res.append(path[:])
            return
        for i in range(start, len(cands)):
            if cands[i] > remain:
                break
            path.append(cands[i])
            go(i, remain - cands[i])
            path.pop()

    go(0, target)
    return res


pruning_lesson()

[[2, 2, 2, 2], [2, 3, 3], [3, 5]]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Hit 8: [2, 2, 2, 2]
2. 3 &gt; remaining 2 → prune, and skip all bigger ones
3. 3 &gt; remaining 1 → prune, and skip all bigger ones
4. 5 &gt; remaining 4 → prune, and skip all bigger ones
5. Hit 8: [2, 3, 3]
6. 5 &gt; remaining 3 → prune, and skip all bigger ones
7. 5 &gt; remaining 1 → prune, and skip all bigger ones
8. 3 &gt; remaining 2 → prune, and skip all bigger ones
9. Hit 8: [3, 5]
10. 5 &gt; remaining 3 → prune, and skip all bigger ones

</details>


**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q6](#q6), [Q2](#q2), [Q12](#q12), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 6: Constraint problems

*Level: Advanced*

N-Queens, Sudoku, word search and partitioning problems are backtracking with **constraints**. Two things make them fast:




- **O(1) validity checks**: keep sets of used columns and diagonals (`r − c`, `r + c`) for queens; sets per row/column/box for Sudoku; mark grid cells in place for word search and unmark on return.
- **Smart order**: try the most constrained spot first (the Sudoku cell with the fewest options), place large items first (matchsticks, k-equal subsets), and skip symmetric choices (two empty buckets are interchangeable).



Good pruning can turn "impossible" (9⁶⁰ for Sudoku) into milliseconds.

**✏️ Your turn:** you now have what you need for [Q18](#q18), [Q19](#q19), [Q11](#q11), [Q16](#q16), [Q17](#q17), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 7: From backtracking to memoisation

*Level: Advanced*

Sometimes different branches reach the **same sub-problem**. In word break II, the suffix starting at index 7 is solved again every time a different split reaches index 7.



If the answer for a sub-problem depends only on its arguments (here, the start index), store it in a dict the first time and reuse it — **memoisation**:



```python
memo = {}
def go(i):
    if i in memo: return memo[i]
    ...
    memo[i] = answer
    return answer
```



That's exactly the doorway to **dynamic programming**. Ask yourself: "is the result determined by a small state, and does that state repeat?" If yes, memoise.

**✏️ Your turn:** you now have what you need for [Q20](#q20), [Q10](#q10). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [5]:
SORT2 = lambda r: sorted(sorted(x) for x in r)


<a id="q1"></a>
### Q1. Subsets

🟡 Interview standard · *Include / skip*

Return all subsets (the power set) of distinct integers `nums`.

**Examples**

- `nums = [1, 2, 3]` → `[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]`
- `nums = [0]` → `[[], [0]]`

In [ ]:
def subsets(nums):
    """
    Return all subsets (the power set) of distinct integers nums.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(subsets, [(([1, 2, 3],), [[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]), (([0],), [[], [0]])], norm=SORT2)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count a bitmask from 0 to 2ⁿ − 1; bit i set means nums[i] is in the subset.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Backtracking with a start index: every node of the tree is a subset, so record the path on entry, then extend it with each later element.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Count a bitmask from 0 to 2ⁿ − 1; bit i set means nums[i] is in the subset.

```python
def subsets(nums):
    n = len(nums)
    return [[nums[i] for i in range(n) if mask >> i & 1] for mask in range(1 << n)]
```

**Cost:** O(n · 2ⁿ) time, O(n · 2ⁿ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Backtracking with a start index: every node of the tree is a subset, so record the path on entry, then extend it with each later element.

```python
def subsets(nums):
    res, path = [], []

    def go(start):
        res.append(path[:])
        for i in range(start, len(nums)):
            path.append(nums[i])
            go(i + 1)
            path.pop()

    go(0)
    return res
```

**Cost:** O(n · 2ⁿ) time, O(n) + output space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count a bitmask from 0 to 2ⁿ − 1; bit i set means nums[i] is in the subset. That costs O(n · 2ⁿ) time and O(n · 2ⁿ) space."

"I can do better: Backtracking with a start index: every node of the tree is a subset, so record the path on entry, then extend it with each later element. That's O(n · 2ⁿ) time and O(n) + output space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 3]`

1. Record []
2. Record [1]
3. Record [1, 2]
4. Record [1, 2, 3]
5. Record [1, 3]
6. Record [2]
7. Record [2, 3]
8. Record [3]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/subsets)

<a id="q2"></a>
### Q2. Subsets II

🟡 Interview standard · *Sort + skip duplicates*

`nums` may contain duplicates. Return all distinct subsets.

**Examples**

- `nums = [1, 2, 2]` → `[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]`
- `nums = [0]` → `[[], [0]]`

In [ ]:
def subsets_with_dup(nums):
    """
    nums may contain duplicates. Return all distinct subsets.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(subsets_with_dup, [(([1, 2, 2],), [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]), (([0],), [[], [0]])], norm=SORT2)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Generate all 2ⁿ subsets, sort each, and deduplicate with a set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort first. At the same tree level, skip nums[i] if it equals nums[i − 1] — choosing either copy leads to identical subsets.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Generate all 2ⁿ subsets, sort each, and deduplicate with a set.

```python
def subsets_with_dup(nums):
    n, seen = len(nums), set()
    for mask in range(1 << n):
        seen.add(tuple(sorted(nums[i] for i in range(n) if mask >> i & 1)))
    return [list(t) for t in seen]
```

**Cost:** O(n · 2ⁿ log n) time, O(n · 2ⁿ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort first. At the same tree level, skip nums[i] if it equals nums[i − 1] — choosing either copy leads to identical subsets.

```python
def subsets_with_dup(nums):
    nums.sort()
    res, path = [], []

    def go(start):
        res.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue
            path.append(nums[i])
            go(i + 1)
            path.pop()

    go(0)
    return res
```

**Cost:** O(n · 2ⁿ) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Generate all 2ⁿ subsets, sort each, and deduplicate with a set. That costs O(n · 2ⁿ log n) time and O(n · 2ⁿ) space."

"I can do better: Sort first. At the same tree level, skip nums[i] if it equals nums[i − 1] — choosing either copy leads to identical subsets. That's O(n · 2ⁿ) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 2]`

1. Take 1 → [1]
2. Take 2 → [1, 2]
3. Take 2 → [1, 2, 2]
4. Skip the second 2 at this level (duplicate branch)
5. Take 2 → [2]
6. Take 2 → [2, 2]
7. Skip the second 2 at this level (duplicate branch)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/subsets-ii)

<a id="q3"></a>
### Q3. Permutations

🟡 Interview standard · *used[] array*

Return all permutations of distinct integers `nums`.

**Examples**

- `nums = [1, 2, 3]` → `[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]`
- `nums = [0, 1]` → `[[0, 1], [1, 0]]`
- `nums = [1]` → `[[1]]`

In [ ]:
def permute(nums):
    """
    Return all permutations of distinct integers nums.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(permute, [(([1, 2, 3],), [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]), (([0, 1],), [[0, 1], [1, 0]]), (([1],), [[1]])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Generate all nⁿ sequences of picks and keep those with no repeated index.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

At each position try every unused element, mark it used, recurse, unmark.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Generate all nⁿ sequences of picks and keep those with no repeated index.

```python
def permute(nums):
    from itertools import product
    n = len(nums)
    return [[nums[i] for i in p] for p in product(range(n), repeat=n) if len(set(p)) == n]
```

**Cost:** O(n · nⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** At each position try every unused element, mark it used, recurse, unmark.

```python
def permute(nums):
    res, path, used = [], [], [False] * len(nums)

    def go():
        if len(path) == len(nums):
            res.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True
            path.append(nums[i])
            go()
            path.pop()
            used[i] = False

    go()
    return res
```

**Cost:** O(n · n!) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Generate all nⁿ sequences of picks and keep those with no repeated index. That costs O(n · nⁿ) time and O(n) space."

"I can do better: At each position try every unused element, mark it used, recurse, unmark. That's O(n · n!) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 2, 3]`

1. Full permutation [1, 2, 3]
2. Full permutation [1, 3, 2]
3. Full permutation [2, 1, 3]
4. Full permutation [2, 3, 1]
5. Full permutation [3, 1, 2]
6. Full permutation [3, 2, 1]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/permutations)

<a id="q4"></a>
### Q4. Permutations II

🟡 Interview standard · *Sort + skip used duplicates*

`nums` may contain duplicates. Return all distinct permutations.

**Examples**

- `nums = [1, 1, 2]` → `[[1, 1, 2], [1, 2, 1], [2, 1, 1]]`
- `nums = [1, 2, 3]` → `[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]`

In [ ]:
def permute_unique(nums):
    """
    nums may contain duplicates. Return all distinct permutations.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(permute_unique, [(([1, 1, 2],), [[1, 1, 2], [1, 2, 1], [2, 1, 1]]), (([1, 2, 3],), [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Generate every permutation and deduplicate with a set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort, then at each position skip nums[i] when it equals nums[i − 1] and nums[i − 1] is not currently used — that forces equal values to be placed in order.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Generate every permutation and deduplicate with a set.

```python
def permute_unique(nums):
    out = set()

    def go(path, rest):
        if not rest:
            out.add(tuple(path))
        for i in range(len(rest)):
            go(path + [rest[i]], rest[:i] + rest[i + 1:])
    go([], nums)
    return [list(p) for p in out]
```

**Cost:** O(n · n!) time, O(n · n!) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort, then at each position skip nums[i] when it equals nums[i − 1] and nums[i − 1] is not currently used — that forces equal values to be placed in order.

```python
def permute_unique(nums):
    nums.sort()
    res, path, used = [], [], [False] * len(nums)

    def go():
        if len(path) == len(nums):
            res.append(path[:])
            return
        for i in range(len(nums)):
            if used[i] or (i > 0 and nums[i] == nums[i - 1] and not used[i - 1]):
                continue
            used[i] = True
            path.append(nums[i])
            go()
            path.pop()
            used[i] = False

    go()
    return res
```

**Cost:** O(n · n!) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Generate every permutation and deduplicate with a set. That costs O(n · n!) time and O(n · n!) space."

"I can do better: Sort, then at each position skip nums[i] when it equals nums[i − 1] and nums[i − 1] is not currently used — that forces equal values to be placed in order. That's O(n · n!) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [1, 1, 2]`

1. Found [1, 1, 2]
2. Found [1, 2, 1]
3. Found [2, 1, 1]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/permutations-ii)

<a id="q5"></a>
### Q5. Combination Sum

🟡 Interview standard · *Reuse allowed · prune by sum*

Return all unique combinations of `candidates` (distinct, each reusable) that sum to `target`.

**Examples**

- `candidates = [2, 3, 6, 7], target = 7` → `[[2, 2, 3], [7]]`
- `candidates = [2, 3, 5], target = 8` → `[[2, 2, 2, 2], [2, 3, 3], [3, 5]]`
- `candidates = [2], target = 1` → `[]`

In [ ]:
def combination_sum(candidates, target):
    """
    Return all unique combinations of candidates (distinct, each reusable) that
    sum to target.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(combination_sum, [(([2, 3, 6, 7], 7), [[2, 2, 3], [7]]), (([2, 3, 5], 8), [[2, 2, 2, 2], [2, 3, 3], [3, 5]]), (([2], 1), [])], norm=SORT2)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build every ordered sequence that doesn't exceed target, then sort each and deduplicate.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort; recurse with start index i (not i + 1) so a number can repeat but order never goes backwards. Stop the loop once a candidate exceeds the remaining sum.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Build every ordered sequence that doesn't exceed target, then sort each and deduplicate.

```python
def combination_sum(candidates, target):
    found = set()

    def go(path, total):
        if total == target:
            found.add(tuple(sorted(path)))
            return
        for c in candidates:
            if total + c <= target:
                go(path + [c], total + c)
    go([], 0)
    return [list(t) for t in found]
```

**Cost:** O(n^(t/min)) time, O(t/min) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort; recurse with start index i (not i + 1) so a number can repeat but order never goes backwards. Stop the loop once a candidate exceeds the remaining sum.

```python
def combination_sum(candidates, target):
    candidates.sort()
    res, path = [], []

    def go(start, remain):
        if remain == 0:
            res.append(path[:])
            return
        for i in range(start, len(candidates)):
            if candidates[i] > remain:
                break
            path.append(candidates[i])
            go(i, remain - candidates[i])
            path.pop()

    go(0, target)
    return res
```

**Cost:** O(n^(t/min)) worst, heavily pruned time, O(t/min) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build every ordered sequence that doesn't exceed target, then sort each and deduplicate. That costs O(n^(t/min)) time and O(t/min) space."

"I can do better: Sort; recurse with start index i (not i + 1) so a number can repeat but order never goes backwards. Stop the loop once a candidate exceeds the remaining sum. That's O(n^(t/min)) worst, heavily pruned time and O(t/min) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `candidates = [2, 3, 6, 7], target = 7`

1. 2 &gt; remaining 1 → prune the rest
2. Sum hit 7: [2, 2, 3]
3. 6 &gt; remaining 3 → prune the rest
4. 3 &gt; remaining 2 → prune the rest
5. 6 &gt; remaining 5 → prune the rest
6. 3 &gt; remaining 1 → prune the rest
7. 6 &gt; remaining 4 → prune the rest
8. 6 &gt; remaining 1 → prune the rest
9. Sum hit 7: [7]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/combination-sum)

<a id="q6"></a>
### Q6. Combination Sum II

🟡 Interview standard · *Each once · skip duplicates*

Each candidate may be used once and the input may contain duplicates. Return unique combinations summing to `target`.

**Examples**

- `candidates = [10, 1, 2, 7, 6, 1, 5], target = 8` → `[[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]`
- `candidates = [2, 5, 2, 1, 2], target = 5` → `[[1, 2, 2], [5]]`

In [ ]:
def combination_sum2(candidates, target):
    """
    Each candidate may be used once and the input may contain duplicates. Return
    unique combinations summing to target.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(combination_sum2, [(([10, 1, 2, 7, 6, 1, 5], 8), [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]), (([2, 5, 2, 1, 2], 5), [[1, 2, 2], [5]])], norm=SORT2)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try all 2ⁿ subsets, keep those that hit target, deduplicate sorted tuples.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort; move to i + 1 after using candidates[i]; skip equal values at the same level; break when a value exceeds the remaining sum.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try all 2ⁿ subsets, keep those that hit target, deduplicate sorted tuples.

```python
def combination_sum2(candidates, target):
    n, found = len(candidates), set()
    for mask in range(1 << n):
        pick = [candidates[i] for i in range(n) if mask >> i & 1]
        if sum(pick) == target:
            found.add(tuple(sorted(pick)))
    return [list(t) for t in found]
```

**Cost:** O(n · 2ⁿ) time, O(n · 2ⁿ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort; move to i + 1 after using candidates[i]; skip equal values at the same level; break when a value exceeds the remaining sum.

```python
def combination_sum2(candidates, target):
    candidates.sort()
    res, path = [], []

    def go(start, remain):
        if remain == 0:
            res.append(path[:])
            return
        for i in range(start, len(candidates)):
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            if candidates[i] > remain:
                break
            path.append(candidates[i])
            go(i + 1, remain - candidates[i])
            path.pop()

    go(0, target)
    return res
```

**Cost:** O(2ⁿ) worst, pruned time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try all 2ⁿ subsets, keep those that hit target, deduplicate sorted tuples. That costs O(n · 2ⁿ) time and O(n · 2ⁿ) space."

"I can do better: Sort; move to i + 1 after using candidates[i]; skip equal values at the same level; break when a value exceeds the remaining sum. That's O(2ⁿ) worst, pruned time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `candidates = [2, 1, 2, 3, 1], target = 4`

1. Take 1, remaining 3
2. Take 1, remaining 2
3. Take 2, remaining 0
4. Found [1, 1, 2]
5. Take 2, remaining 1
6. Take 3, remaining 0
7. Found [1, 3]
8. Take 2, remaining 2
9. Take 2, remaining 0
10. Found [2, 2]
11. Take 3, remaining 1

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/combination-sum-ii)

<a id="q7"></a>
### Q7. Combinations

🟡 Interview standard · *Choose k · prune by count*

Return all combinations of `k` numbers chosen from 1..n.

**Examples**

- `n = 4, k = 2` → `[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]`
- `n = 1, k = 1` → `[[1]]`

In [ ]:
def combine(n, k):
    """
    Return all combinations of k numbers chosen from 1..n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(combine, [((4, 2), [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]), ((1, 1), [[1]])], norm=SORT2)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check all 2ⁿ bitmasks and keep those with exactly k bits.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Backtrack from 'start'. Prune when there aren't enough numbers left: loop only up to n − (k − len(path)) + 1.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check all 2ⁿ bitmasks and keep those with exactly k bits.

```python
def combine(n, k):
    return [[i + 1 for i in range(n) if m >> i & 1] for m in range(1 << n) if bin(m).count("1") == k]
```

**Cost:** O(n · 2ⁿ) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Backtrack from 'start'. Prune when there aren't enough numbers left: loop only up to n − (k − len(path)) + 1.

```python
def combine(n, k):
    res, path = [], []

    def go(start):
        if len(path) == k:
            res.append(path[:])
            return
        need = k - len(path)
        for x in range(start, n - need + 2):
            path.append(x)
            go(x + 1)
            path.pop()

    go(1)
    return res
```

**Cost:** O(k · C(n, k)) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check all 2ⁿ bitmasks and keep those with exactly k bits. That costs O(n · 2ⁿ) time and O(k) space."

"I can do better: Backtrack from 'start'. Prune when there aren't enough numbers left: loop only up to n − (k − len(path)) + 1. That's O(k · C(n, k)) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 5, k = 3`

1. Pick [1, 2, 3]
2. Pick [1, 2, 4]
3. Pick [1, 2, 5]
4. Pick [1, 3, 4]
5. Pick [1, 3, 5]
6. Pick [1, 4, 5]
7. Pick [2, 3, 4]
8. Pick [2, 3, 5]
9. Pick [2, 4, 5]
10. Pick [3, 4, 5]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/combinations)

<a id="q8"></a>
### Q8. Letter Combinations of a Phone Number

🟡 Interview standard · *One level per digit*

Given digits 2–9, return all letter combinations they could represent on a phone keypad.

**Examples**

- `digits = '23'` → `['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']`
- `digits = ''` → `[]`
- `digits = '2'` → `['a', 'b', 'c']`

In [ ]:
def letter_combinations(digits):
    """
    Given digits 2–9, return all letter combinations they could represent on a
    phone keypad.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(letter_combinations, [(("23",), ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]), (("",), []), (("2",), ["a", "b", "c"])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Cartesian product of the letter groups with itertools.product.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Backtracking: level i chooses one letter for digits[i]; at depth n record the string.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Cartesian product of the letter groups with itertools.product.

```python
def letter_combinations(digits):
    PHONE = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
    from itertools import product
    if not digits:
        return []
    return ["".join(p) for p in product(*(PHONE[d] for d in digits))]
```

**Cost:** O(4ⁿ · n) time, O(4ⁿ · n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Backtracking: level i chooses one letter for digits[i]; at depth n record the string.

```python
def letter_combinations(digits):
    PHONE = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl", "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
    if not digits:
        return []
    res, path = [], []

    def go(i):
        if i == len(digits):
            res.append("".join(path))
            return
        for ch in PHONE[digits[i]]:
            path.append(ch)
            go(i + 1)
            path.pop()

    go(0)
    return res
```

**Cost:** O(4ⁿ · n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Cartesian product of the letter groups with itertools.product. That costs O(4ⁿ · n) time and O(4ⁿ · n) space."

"I can do better: Backtracking: level i chooses one letter for digits[i]; at depth n record the string. That's O(4ⁿ · n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `digits = '23'`

1. Record 'ad'
2. Record 'ae'
3. Record 'af'
4. Record 'bd'
5. Record 'be'
6. Record 'bf'
7. Record 'cd'
8. Record 'ce'
9. Record 'cf'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/letter-combinations-of-a-phone-number)

<a id="q9"></a>
### Q9. Generate Parentheses

🟡 Interview standard · *Open / close counts*

Return all well-formed strings of `n` pairs of parentheses.

**Examples**

- `n = 3` → `['((()))', '(()())', '(())()', '()(())', '()()()']`
- `n = 1` → `['()']`

In [ ]:
def generate_parenthesis(n):
    """
    Return all well-formed strings of n pairs of parentheses.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(generate_parenthesis, [((3,), ["((()))", "(()())", "(())()", "()(())", "()()()"]), ((1,), ["()"])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Generate all 2²ⁿ strings of '(' and ')' and keep the balanced ones.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Add '(' while open &lt; n; add ')' only while close &lt; open. Every branch stays valid, so nothing is generated and thrown away.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Generate all 2²ⁿ strings of '(' and ')' and keep the balanced ones.

```python
def generate_parenthesis(n):
    from itertools import product

    def ok(s):
        bal = 0
        for c in s:
            bal += 1 if c == "(" else -1
            if bal < 0:
                return False
        return bal == 0
    return ["".join(p) for p in product("()", repeat=2 * n) if ok(p)]
```

**Cost:** O(n · 2²ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Add '(' while open &lt; n; add ')' only while close &lt; open. Every branch stays valid, so nothing is generated and thrown away.

```python
def generate_parenthesis(n):
    res, path = [], []

    def go(open_, close):
        if len(path) == 2 * n:
            res.append("".join(path))
            return
        if open_ < n:
            path.append("(")
            go(open_ + 1, close)
            path.pop()
        if close < open_:
            path.append(")")
            go(open_, close + 1)
            path.pop()

    go(0, 0)
    return res
```

**Cost:** O(4ⁿ / √n) — the Catalan number time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Generate all 2²ⁿ strings of '(' and ')' and keep the balanced ones. That costs O(n · 2²ⁿ) time and O(n) space."

"I can do better: Add '(' while open &lt; n; add ')' only while close &lt; open. Every branch stays valid, so nothing is generated and thrown away. That's O(4ⁿ / √n) — the Catalan number time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 2`

1. Add '(' (open 1)
2. Add '(' (open 2)
3. Add ')' (close 1)
4. Add ')' (close 2)
5. Complete: (())
6. Add ')' (close 1)
7. Add '(' (open 2)
8. Add ')' (close 2)
9. Complete: ()()

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/generate-parentheses)

<a id="q10"></a>
### Q10. Palindrome Partitioning

🟡 Interview standard · *Cut where the prefix is a palindrome*

Split `s` so every piece is a palindrome. Return all such partitions.

**Examples**

- `s = 'aab'` → `[['a', 'a', 'b'], ['aa', 'b']]`
- `s = 'a'` → `[['a']]`

In [ ]:
def partition_pal(s):
    """
    Split s so every piece is a palindrome. Return all such partitions.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(partition_pal, [(("aab",), [["a", "a", "b"], ["aa", "b"]]), (("a",), [["a"]])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try all 2ⁿ⁻¹ ways to place cuts and keep those whose pieces are all palindromes.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

From index i, try every end j where s[i..j] is a palindrome (precomputed in a DP table), and recurse from j + 1.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try all 2ⁿ⁻¹ ways to place cuts and keep those whose pieces are all palindromes.

```python
def partition_pal(s):
    n, res = len(s), []
    for mask in range(1 << max(0, n - 1)):
        parts, start = [], 0
        for i in range(n - 1):
            if mask >> i & 1:
                parts.append(s[start:i + 1])
                start = i + 1
        parts.append(s[start:])
        if all(p == p[::-1] for p in parts):
            res.append(parts)
    return res
```

**Cost:** O(n · 2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** From index i, try every end j where s[i..j] is a palindrome (precomputed in a DP table), and recurse from j + 1.

```python
def partition_pal(s):
    n = len(s)
    pal = [[False] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            pal[i][j] = s[i] == s[j] and (j - i < 2 or pal[i + 1][j - 1])
    res, path = [], []

    def go(i):
        if i == n:
            res.append(path[:])
            return
        for j in range(i, n):
            if pal[i][j]:
                path.append(s[i:j + 1])
                go(j + 1)
                path.pop()

    go(0)
    return res
```

**Cost:** O(n · 2ⁿ) time, O(n²) table space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try all 2ⁿ⁻¹ ways to place cuts and keep those whose pieces are all palindromes. That costs O(n · 2ⁿ) time and O(n) space."

"I can do better: From index i, try every end j where s[i..j] is a palindrome (precomputed in a DP table), and recurse from j + 1. That's O(n · 2ⁿ) time and O(n²) table space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'aabb'`

1. 'a' is a palindrome → cut after index 0
2. 'a' is a palindrome → cut after index 1
3. 'b' is a palindrome → cut after index 2
4. 'b' is a palindrome → cut after index 3
5. Partition ['a', 'a', 'b', 'b']
6. 'bb' is a palindrome → cut after index 3
7. Partition ['a', 'a', 'bb']
8. 'aa' is a palindrome → cut after index 1
9. 'b' is a palindrome → cut after index 2
10. 'b' is a palindrome → cut after index 3
11. Partition ['aa', 'b', 'b']
12. 'bb' is a palindrome → cut after index 3
13. Partition ['aa', 'bb']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/palindrome-partitioning)

<a id="q11"></a>
### Q11. Word Search

🟡 Interview standard · *Grid DFS · mark and unmark*

Can `word` be traced in the grid through horizontally or vertically adjacent cells, using each cell at most once?

**Examples**

- `board = [['A', 'B', 'C', 'E'], ['S', 'F', 'C', 'S'], ['A', 'D', 'E', 'E']], word = 'ABCCED'` → `True`
- `board = [['A', 'B', 'C', 'E'], ['S', 'F', 'C', 'S'], ['A', 'D', 'E', 'E']], word = 'SEE'` → `True`
- `board = [['A', 'B', 'C', 'E'], ['S', 'F', 'C', 'S'], ['A', 'D', 'E', 'E']], word = 'ABCB'` → `False`

In [ ]:
def exist(board, word):
    """
    Can word be traced in the grid through horizontally or vertically adjacent
    cells, using each cell at most once?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(exist, [
    (([['A', 'B', 'C', 'E'], ['S', 'F', 'C', 'S'], ['A', 'D', 'E', 'E']], 'ABCCED'), True),
    (([['A', 'B', 'C', 'E'], ['S', 'F', 'C', 'S'], ['A', 'D', 'E', 'E']], 'SEE'), True),
    (([['A', 'B', 'C', 'E'], ['S', 'F', 'C', 'S'], ['A', 'D', 'E', 'E']], 'ABCB'), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

DFS from every cell, passing a fresh copy of the visited set on each step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

DFS from each cell that matches word[0]. Mark the cell in place while it's on the path, restore it on return. Quick prune: if the board lacks enough of some letter, return False up front.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** DFS from every cell, passing a fresh copy of the visited set on each step.

```python
def exist(board, word):
    m, n = len(board), len(board[0])

    def dfs(r, c, i, seen):
        if i == len(word):
            return True
        if not (0 <= r < m and 0 <= c < n) or (r, c) in seen or board[r][c] != word[i]:
            return False
        seen = seen | {(r, c)}
        return any(dfs(r + dr, c + dc, i + 1, seen) for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))
    return any(dfs(r, c, 0, set()) for r in range(m) for c in range(n))
```

**Cost:** O(m · n · 4ᴸ · L) time, O(L²) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** DFS from each cell that matches word[0]. Mark the cell in place while it's on the path, restore it on return. Quick prune: if the board lacks enough of some letter, return False up front.

```python
def exist(board, word):
    from collections import Counter
    m, n = len(board), len(board[0])
    have = Counter(ch for row in board for ch in row)
    if any(have[ch] < cnt for ch, cnt in Counter(word).items()):
        return False
    path = []

    def dfs(r, c, i):
        if i == len(word):
            return True
        if not (0 <= r < m and 0 <= c < n) or board[r][c] != word[i]:
            return False
        board[r][c] = "#"
        path.append((r, c))
        found = any(dfs(r + dr, c + dc, i + 1) for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))
        board[r][c] = word[i]
        path.pop()
        return found

    return any(dfs(r, c, 0) for r in range(m) for c in range(n))
```

**Cost:** O(m · n · 3ᴸ) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: DFS from every cell, passing a fresh copy of the visited set on each step. That costs O(m · n · 4ᴸ · L) time and O(L²) space."

"I can do better: DFS from each cell that matches word[0]. Mark the cell in place while it's on the path, restore it on return. Quick prune: if the board lacks enough of some letter, return False up front. That's O(m · n · 3ᴸ) time and O(L) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `board = [['A', 'B', 'C', 'E'], ['S', 'F', 'C', 'S'], ['A', 'D', 'E', 'E']], word = 'SEE'`

1. Matched 'S' at (1,0)
2. Matched 'S' at (1,3)
3. Matched 'E' at (2,3)
4. Matched 'E' at (2,2)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/word-search)

<a id="q12"></a>
### Q12. Combination Sum III

🟡 Interview standard · *Choose k from 1..9*

Find all combinations of `k` distinct numbers from 1–9 that sum to `n`.

**Examples**

- `k = 3, n = 7` → `[[1, 2, 4]]`
- `k = 3, n = 9` → `[[1, 2, 6], [1, 3, 5], [2, 3, 4]]`
- `k = 4, n = 1` → `[]`

In [ ]:
def combination_sum3(k, n):
    """
    Find all combinations of k distinct numbers from 1–9 that sum to n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(combination_sum3, [((3, 7), [[1, 2, 4]]), ((3, 9), [[1, 2, 6], [1, 3, 5], [2, 3, 4]]), ((4, 1), [])], norm=SORT2)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check all 2⁹ subsets of 1..9 for size k and sum n.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Backtrack from 'start', stopping when the remaining sum goes negative or the path is already k long.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Check all 2⁹ subsets of 1..9 for size k and sum n.

```python
def combination_sum3(k, n):
    return [[d + 1 for d in range(9) if m >> d & 1] for m in range(1 << 9)
            if bin(m).count("1") == k and sum(d + 1 for d in range(9) if m >> d & 1) == n]
```

**Cost:** O(9 · 2⁹) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Backtrack from 'start', stopping when the remaining sum goes negative or the path is already k long.

```python
def combination_sum3(k, n):
    res, path = [], []

    def go(start, remain):
        if len(path) == k:
            if remain == 0:
                res.append(path[:])
            return
        for x in range(start, 10):
            if x > remain:
                break
            path.append(x)
            go(x + 1, remain - x)
            path.pop()

    go(1, n)
    return res
```

**Cost:** O(C(9, k) · k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check all 2⁹ subsets of 1..9 for size k and sum n. That costs O(9 · 2⁹) time and O(k) space."

"I can do better: Backtrack from 'start', stopping when the remaining sum goes negative or the path is already k long. That's O(C(9, k) · k) time and O(k) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `k = 3, n = 9`

1. Found [1, 2, 6]
2. Found [1, 3, 5]
3. Found [2, 3, 4]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/combination-sum-iii)

<a id="q13"></a>
### Q13. Restore IP Addresses

🟡 Interview standard · *Four parts with validity pruning*

Insert three dots into the digit string `s` to form all valid IPv4 addresses (each part 0–255, no leading zeros).

**Examples**

- `s = '25525511135'` → `['255.255.11.135', '255.255.111.35']`
- `s = '0000'` → `['0.0.0.0']`
- `s = '101023'` → `['1.0.10.23', '1.0.102.3', '10.1.0.23', '10.10.2.3', '101.0.2.3']`

In [ ]:
def restore_ip_addresses(s):
    """
    Insert three dots into the digit string s to form all valid IPv4 addresses
    (each part 0–255, no leading zeros).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(restore_ip_addresses, [(("25525511135",), ["255.255.11.135", "255.255.111.35"]), (("0000",), ["0.0.0.0"]), (("101023",), ["1.0.10.23", "1.0.102.3", "10.1.0.23", "10.10.2.3", "101.0.2.3"])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every way to place 3 dots among the gaps and validate the four parts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Backtrack part by part, taking 1–3 digits. Prune when the rest is too short or too long for the remaining parts, or the part is invalid.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every way to place 3 dots among the gaps and validate the four parts.

```python
def restore_ip_addresses(s):
    def valid(p):
        return len(p) > 0 and (p == "0" or p[0] != "0") and int(p) <= 255

    n, res = len(s), []
    for a in range(1, n):
        for b in range(a + 1, n):
            for c in range(b + 1, n):
                parts = [s[:a], s[a:b], s[b:c], s[c:]]
                if all(len(p) <= 3 and valid(p) for p in parts):
                    res.append(".".join(parts))
    return res
```

**Cost:** O(n³) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Backtrack part by part, taking 1–3 digits. Prune when the rest is too short or too long for the remaining parts, or the part is invalid.

```python
def restore_ip_addresses(s):
    def valid(p):
        return len(p) > 0 and (p == "0" or p[0] != "0") and int(p) <= 255

    res, parts = [], []

    def go(i):
        left = 4 - len(parts)
        if len(s) - i < left or len(s) - i > 3 * left:
            return
        if left == 0:
            res.append(".".join(parts))
            return
        for size in (1, 2, 3):
            part = s[i:i + size]
            if len(part) == size and valid(part):
                parts.append(part)
                go(i + size)
                parts.pop()

    go(0)
    return res
```

**Cost:** O(3⁴) = O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every way to place 3 dots among the gaps and validate the four parts. That costs O(n³) time and O(n) space."

"I can do better: Backtrack part by part, taking 1–3 digits. Prune when the rest is too short or too long for the remaining parts, or the part is invalid. That's O(3⁴) = O(1) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = '101023'`

1. Valid address 1.0.10.23
2. Valid address 1.0.102.3
3. Valid address 10.1.0.23
4. Valid address 10.10.2.3
5. Valid address 101.0.2.3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/restore-ip-addresses)

<a id="q14"></a>
### Q14. Pow(x, n)

🟡 Interview standard · *Fast exponentiation*

Compute x raised to the integer power n (n may be negative).

**Examples**

- `x = 2.0, n = 10` → `1024.0`
- `x = 2.1, n = 3` → `9.261`
- `x = 2.0, n = -2` → `0.25`

In [ ]:
def my_pow(x, n):
    """
    Compute x raised to the integer power n (n may be negative).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(my_pow, [((2.0, 10), 1024.0), ((2.1, 3), 9.261), ((2.0, -2), 0.25)], norm=lambda v: round(v, 6))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Multiply x by itself |n| times.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

xⁿ = (x²)^(n/2) for even n, and x · xⁿ⁻¹ for odd n. Square the base and halve the exponent each step.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Multiply x by itself |n| times.

```python
def my_pow(x, n):
    res = 1.0
    for _ in range(abs(n)):
        res *= x
    return res if n >= 0 else 1 / res
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** xⁿ = (x²)^(n/2) for even n, and x · xⁿ⁻¹ for odd n. Square the base and halve the exponent each step.

```python
def my_pow(x, n):
    if n < 0:
        x, n = 1 / x, -n
    res = 1.0
    while n:
        if n & 1:
            res *= x
        x *= x
        n >>= 1
    return res
```

**Cost:** O(log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Multiply x by itself |n| times. That costs O(n) time and O(1) space."

"I can do better: xⁿ = (x²)^(n/2) for even n, and x · xⁿ⁻¹ for odd n. Square the base and halve the exponent each step. That's O(log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `x = 3.0, n = 13`

1. n = 13 (0b1101): odd → res ×= base; then square base
2. n = 6 (0b110): even; then square base
3. n = 3 (0b11): odd → res ×= base; then square base
4. n = 1 (0b1): odd → res ×= base; then square base

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/pow-x-n)

<a id="q15"></a>
### Q15. Letter Case Permutation

🟡 Interview standard · *Branch on letters only*

Return every string obtainable by changing the case of each letter in `s`.

**Examples**

- `s = 'a1b2'` → `['a1b2', 'a1B2', 'A1b2', 'A1B2']`
- `s = '3z4'` → `['3z4', '3Z4']`

In [ ]:
def letter_case_permutation(s):
    """
    Return every string obtainable by changing the case of each letter in s.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(letter_case_permutation, [(("a1b2",), ["a1b2", "a1B2", "A1b2", "A1B2"]), (("3z4",), ["3z4", "3Z4"])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over a bitmask of the letters; bit i decides the case of letter i.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Backtrack through s; digits have one branch, letters have two (lower, upper).

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Loop over a bitmask of the letters; bit i decides the case of letter i.

```python
def letter_case_permutation(s):
    idx = [i for i, ch in enumerate(s) if ch.isalpha()]
    res = []
    for m in range(1 << len(idx)):
        cur = list(s.lower())
        for b, i in enumerate(idx):
            if m >> b & 1:
                cur[i] = cur[i].upper()
        res.append("".join(cur))
    return res
```

**Cost:** O(n · 2ᴸ) time, O(n · 2ᴸ) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Backtrack through s; digits have one branch, letters have two (lower, upper).

```python
def letter_case_permutation(s):
    res, path = [], []

    def go(i):
        if i == len(s):
            res.append("".join(path))
            return
        options = {s[i].lower(), s[i].upper()} if s[i].isalpha() else {s[i]}
        for ch in sorted(options):
            path.append(ch)
            go(i + 1)
            path.pop()

    go(0)
    return res
```

**Cost:** O(n · 2ᴸ) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over a bitmask of the letters; bit i decides the case of letter i. That costs O(n · 2ᴸ) time and O(n · 2ᴸ) space."

"I can do better: Backtrack through s; digits have one branch, letters have two (lower, upper). That's O(n · 2ᴸ) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'a1b2'`

1. Record 'A1B2'
2. Record 'A1b2'
3. Record 'a1B2'
4. Record 'a1b2'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/letter-case-permutation)

<a id="q16"></a>
### Q16. Partition to K Equal Sum Subsets

🟡 Interview standard · *Fill buckets · prune*

Can `nums` be split into `k` non-empty subsets with equal sums?

**Examples**

- `nums = [4, 3, 2, 3, 5, 2, 1], k = 4` → `True`
- `nums = [1, 2, 3, 4], k = 3` → `False`

In [ ]:
def can_partition_k_subsets(nums, k):
    """
    Can nums be split into k non-empty subsets with equal sums?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(can_partition_k_subsets, [
    (([4, 3, 2, 3, 5, 2, 1], 4), True),
    (([1, 2, 3, 4], 3), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Assign each number to one of k buckets in every possible way (kⁿ).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

target = sum / k. Sort descending, place each number into a bucket with room. Skip a bucket whose current sum equals one already tried (symmetry), and stop at the first empty bucket that fails.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Assign each number to one of k buckets in every possible way (kⁿ).

```python
def can_partition_k_subsets(nums, k):
    from itertools import product
    total = sum(nums)
    if total % k:
        return False
    for assign in product(range(k), repeat=len(nums)):
        sums = [0] * k
        for x, b in zip(nums, assign):
            sums[b] += x
        if len(set(sums)) == 1:
            return True
    return False
```

**Cost:** O(kⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** target = sum / k. Sort descending, place each number into a bucket with room. Skip a bucket whose current sum equals one already tried (symmetry), and stop at the first empty bucket that fails.

```python
def can_partition_k_subsets(nums, k):
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    nums.sort(reverse=True)
    if nums[0] > target:
        return False
    buckets = [0] * k

    def go(i):
        if i == len(nums):
            return True
        tried = set()
        for b in range(k):
            if buckets[b] + nums[i] > target or buckets[b] in tried:
                continue
            tried.add(buckets[b])
            buckets[b] += nums[i]
            if go(i + 1):
                return True
            buckets[b] -= nums[i]
        return False

    return go(0)
```

**Cost:** O(kⁿ) worst, heavily pruned time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Assign each number to one of k buckets in every possible way (kⁿ). That costs O(kⁿ) time and O(n) space."

"I can do better: target = sum / k. Sort descending, place each number into a bucket with room. Skip a bucket whose current sum equals one already tried (symmetry), and stop at the first empty bucket that fails. That's O(kⁿ) worst, heavily pruned time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [4, 3, 2, 3, 5, 2, 1], k = 4`

1. Put 5 into bucket 0 (target 5)
2. Put 4 into bucket 1 (target 5)
3. Put 3 into bucket 2 (target 5)
4. Put 3 into bucket 3 (target 5)
5. Put 2 into bucket 2 (target 5)
6. Put 2 into bucket 3 (target 5)
7. Put 1 into bucket 1 (target 5)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/partition-to-k-equal-sum-subsets)

<a id="q17"></a>
### Q17. Matchsticks to Square

🟡 Interview standard · *Four sides · sort + prune*

Can all matchsticks be used (each once, unbroken) to form a square?

**Examples**

- `sticks = [1, 1, 2, 2, 2]` → `True`
- `sticks = [3, 3, 3, 3, 4]` → `False`

In [ ]:
def makesquare(sticks):
    """
    Can all matchsticks be used (each once, unbroken) to form a square?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(makesquare, [
    (([1, 1, 2, 2, 2],), True),
    (([3, 3, 3, 3, 4],), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Assign each stick to one of 4 sides in all 4ⁿ ways.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Side = total / 4. Place sticks longest first into sides with room; skip a side whose length equals one already tried for this stick.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Assign each stick to one of 4 sides in all 4ⁿ ways.

```python
def makesquare(sticks):
    from itertools import product
    total = sum(sticks)
    if total % 4:
        return False
    for assign in product(range(4), repeat=len(sticks)):
        sides = [0] * 4
        for x, b in zip(sticks, assign):
            sides[b] += x
        if len(set(sides)) == 1:
            return True
    return False
```

**Cost:** O(4ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Side = total / 4. Place sticks longest first into sides with room; skip a side whose length equals one already tried for this stick.

```python
def makesquare(sticks):
    total = sum(sticks)
    if total % 4 or not sticks:
        return False
    side = total // 4
    sticks.sort(reverse=True)
    sides = [0] * 4

    def go(i):
        if i == len(sticks):
            return True
        for b in range(4):
            if sides[b] + sticks[i] > side or sides[b] in sides[:b]:
                continue
            sides[b] += sticks[i]
            if go(i + 1):
                return True
            sides[b] -= sticks[i]
        return False

    return go(0)
```

**Cost:** O(4ⁿ) worst, heavily pruned time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Assign each stick to one of 4 sides in all 4ⁿ ways. That costs O(4ⁿ) time and O(n) space."

"I can do better: Side = total / 4. Place sticks longest first into sides with room; skip a side whose length equals one already tried for this stick. That's O(4ⁿ) worst, heavily pruned time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `sticks = [3, 1, 2, 2, 2, 1, 1]`

1. Stick 3 → side 0 (each side must reach 3)
2. Stick 2 → side 1 (each side must reach 3)
3. Stick 2 → side 2 (each side must reach 3)
4. Stick 2 → side 3 (each side must reach 3)
5. Stick 1 → side 1 (each side must reach 3)
6. Stick 1 → side 2 (each side must reach 3)
7. Stick 1 → side 3 (each side must reach 3)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/matchsticks-to-square)

<a id="q18"></a>
### Q18. N-Queens

🔴 Stretch · *Row by row with column / diagonal sets*

Place `n` queens on an n × n board so none attack each other. Return every board.

**Examples**

- `n = 4` → `[['.Q..', '...Q', 'Q...', '..Q.'], ['..Q.', 'Q...', '...Q', '.Q..']]`
- `n = 1` → `[['Q']]`

In [ ]:
def solve_n_queens(n):
    """
    Place n queens on an n × n board so none attack each other. Return every
    board.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(solve_n_queens, [((4,), [[".Q..", "...Q", "Q...", "..Q."], ["..Q.", "Q...", "...Q", ".Q.."]]), ((1,), [["Q"]])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every permutation of columns (one queen per row and column) and check the diagonals.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Place one queen per row. Keep sets of used columns, r − c diagonals and r + c anti-diagonals so each safety check is O(1), and backtrack on conflict.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every permutation of columns (one queen per row and column) and check the diagonals.

```python
def solve_n_queens(n):
    from itertools import permutations
    res = []
    for cols in permutations(range(n)):
        if len({r - c for r, c in enumerate(cols)}) == n and len({r + c for r, c in enumerate(cols)}) == n:
            res.append(["." * c + "Q" + "." * (n - c - 1) for c in cols])
    return res
```

**Cost:** O(n! · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Place one queen per row. Keep sets of used columns, r − c diagonals and r + c anti-diagonals so each safety check is O(1), and backtrack on conflict.

```python
def solve_n_queens(n):
    res, cols = [], []
    used_c, diag, anti = set(), set(), set()

    def board():
        return [["Q" if cols[r] == c else "." for c in range(n)] if r < len(cols) else ["."] * n for r in range(n)]

    def go(r):
        if r == n:
            res.append(["." * c + "Q" + "." * (n - c - 1) for c in cols])
            return
        for c in range(n):
            if c in used_c or r - c in diag or r + c in anti:
                continue
            cols.append(c)
            used_c.add(c); diag.add(r - c); anti.add(r + c)
            go(r + 1)
            cols.pop()
            used_c.remove(c); diag.remove(r - c); anti.remove(r + c)

    go(0)
    return res
```

**Cost:** O(n!) with heavy pruning time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every permutation of columns (one queen per row and column) and check the diagonals. That costs O(n! · n) time and O(n) space."

"I can do better: Place one queen per row. Keep sets of used columns, r − c diagonals and r + c anti-diagonals so each safety check is O(1), and backtrack on conflict. That's O(n!) with heavy pruning time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `n = 4`

1. Queen at row 0, column 0
2. Queen at row 1, column 2
3. Queen at row 1, column 3
4. Queen at row 2, column 1
5. Queen at row 0, column 1
6. Queen at row 1, column 3
7. Queen at row 2, column 0
8. Queen at row 3, column 2
9. Solution #1
10. Queen at row 0, column 2
11. Queen at row 1, column 0
12. Queen at row 2, column 3
13. Queen at row 3, column 1
14. Solution #2
15. Queen at row 0, column 3

*…and so on.*

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/n-queens)

<a id="q19"></a>
### Q19. Sudoku Solver

🔴 Stretch · *Constraint sets + most-constrained cell*

Fill the empty cells ('.') so every row, column and 3 × 3 box contains 1–9 exactly once. Modify the board in place.

**Examples**

- `board = [['5', '3', '.', '.', '7', '.', '.', '.', '.'], ['6', '.', '.', '1', '9', '5', '.', '.', '.'], ['.', '9', '8', '.', '.', '.', '.', '6', '.'], ['8', '.', '.', '.', '6', '.', '.', '.', '3'], ['4', '.', '.', '8', '.', '3', '.', '.', '1'], ['7', '.', '.', '.', '2', '.', '.', '.', '6'], ['.', '6', '.', '.', '.', '.', '2', '8', '.'], ['.', '.', '.', '4', '1', '9', '.', '.', '5'], ['.', '.', '.', '.', '8', '.', '.', '7', '9']]` → `[['5', '3', '4', '6', '7', '8', '9', '1', '2'], ['6', '7', '2', '1', '9', '5', '3', '4', '8'], ['1', '9', '8', '3', '4', '2', '5', '6', '7'], ['8', '5', '9', '7', '6', '1', '4', '2', '3'], ['4', '2', '6', '8', '5', '3', '7', '9', '1'], ['7', '1', '3', '9', '2', '4', '8', '5', '6'], ['9', '6', '1', '5', '3', '7', '2', '8', '4'], ['2', '8', '7', '4', '1', '9', '6', '3', '5'], ['3', '4', '5', '2', '8', '6', '1', '7', '9']]`

In [ ]:
def solve_sudoku(board):
    """
    Fill the empty cells ('.') so every row, column and 3 × 3 box contains 1–9
    exactly once. Modify the board in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(solve_sudoku, [(([["5", "3", ".", ".", "7", ".", ".", ".", "."], ["6", ".", ".", "1", "9", "5", ".", ".", "."], [".", "9", "8", ".", ".", ".", ".", "6", "."], ["8", ".", ".", ".", "6", ".", ".", ".", "3"], ["4", ".", ".", "8", ".", "3", ".", ".", "1"], ["7", ".", ".", ".", "2", ".", ".", ".", "6"], [".", "6", ".", ".", ".", ".", "2", "8", "."], [".", ".", ".", "4", "1", "9", ".", ".", "5"], [".", ".", ".", ".", "8", ".", ".", "7", "9"]],),
    [["5", "3", "4", "6", "7", "8", "9", "1", "2"], ["6", "7", "2", "1", "9", "5", "3", "4", "8"], ["1", "9", "8", "3", "4", "2", "5", "6", "7"], ["8", "5", "9", "7", "6", "1", "4", "2", "3"], ["4", "2", "6", "8", "5", "3", "7", "9", "1"], ["7", "1", "3", "9", "2", "4", "8", "5", "6"], ["9", "6", "1", "5", "3", "7", "2", "8", "4"], ["2", "8", "7", "4", "1", "9", "6", "3", "5"], ["3", "4", "5", "2", "8", "6", "1", "7", "9"]])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fill cells left to right, trying 1–9 and re-scanning the row, column and box to check each digit.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Keep sets of digits used per row, column and box (O(1) checks). Always fill the empty cell with the fewest legal digits next — it fails fast and prunes most of the tree.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Fill cells left to right, trying 1–9 and re-scanning the row, column and box to check each digit.

```python
def solve_sudoku(board):
    def ok(r, c, d):
        for i in range(9):
            if board[r][i] == d or board[i][c] == d or board[r // 3 * 3 + i // 3][c // 3 * 3 + i % 3] == d:
                return False
        return True

    def go(k):
        if k == 81:
            return True
        r, c = divmod(k, 9)
        if board[r][c] != ".":
            return go(k + 1)
        for d in "123456789":
            if ok(r, c, d):
                board[r][c] = d
                if go(k + 1):
                    return True
                board[r][c] = "."
        return False
    go(0)
```

**Cost:** O(9ᵐ · 27) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Keep sets of digits used per row, column and box (O(1) checks). Always fill the empty cell with the fewest legal digits next — it fails fast and prunes most of the tree.

```python
def solve_sudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empty = []
    for r in range(9):
        for c in range(9):
            d = board[r][c]
            if d == ".":
                empty.append((r, c))
            else:
                rows[r].add(d); cols[c].add(d); boxes[r // 3 * 3 + c // 3].add(d)
    filled = []

    def go():
        if not empty:
            return True
        best = max(empty, key=lambda rc: len(rows[rc[0]] | cols[rc[1]] | boxes[rc[0] // 3 * 3 + rc[1] // 3]))  # most digits ruled out
        r, c = best
        b = r // 3 * 3 + c // 3
        empty.remove(best)
        for d in "123456789":
            if d in rows[r] or d in cols[c] or d in boxes[b]:
                continue
            board[r][c] = d
            rows[r].add(d); cols[c].add(d); boxes[b].add(d)
            filled.append((r, c))
            if go():
                return True
            board[r][c] = "."
            rows[r].remove(d); cols[c].remove(d); boxes[b].remove(d)
            filled.pop()
        empty.append(best)
        return False

    go()
```

**Cost:** O(9ᵐ) worst, tiny in practice time, O(81) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fill cells left to right, trying 1–9 and re-scanning the row, column and box to check each digit. That costs O(9ᵐ · 27) time and O(m) space."

"I can do better: Keep sets of digits used per row, column and box (O(1) checks). Always fill the empty cell with the fewest legal digits next — it fails fast and prunes most of the tree. That's O(9ᵐ) worst, tiny in practice time and O(81) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `board = [['5', '3', '.', '.', '7', '.', '.', '.', '.'], ['6', '.', '.', '1', '9', '5', '.', '.', '.'], ['.', '9', '8', '.', '.', '.', '.', '6', '.'], ['8', '.', '.', '.', '6', '.', '.', '.', '3'], ['4', '.', '.', '8', '.', '3', '.', '.', '1'], ['7', '.', '.', '.', '2', '.', '.', '.', '6'], ['.', '6', '.', '.', '.', '.', '2', '8', '.'], ['.', '.', '.', '4', '1', '9', '.', '.', '5'], ['.', '.', '.', '.', '8', '.', '.', '7', '9']]`

1. (4,4) has the fewest options → try 5
2. (4,1) has the fewest options → try 2
3. (4,7) has the fewest options → try 9
4. (4,2) has the fewest options → try 6
5. (4,6) has the fewest options → try 7
6. (5,3) has the fewest options → try 9
7. (3,3) has the fewest options → try 7
8. (6,4) has the fewest options → try 3
9. (2,4) has the fewest options → try 4
10. (2,5) has the fewest options → try 2
11. (0,3) has the fewest options → try 6
12. (0,5) has the fewest options → try 8
13. (2,0) has the fewest options → try 1
14. (2,3) has the fewest options → try 3
15. (2,6) has the fewest options → try 5

*…and so on.*

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/sudoku-solver)

<a id="q20"></a>
### Q20. Word Break II

🔴 Stretch · *Memoised DFS on start index*

Add spaces to `s` to form sentences where every word is in `wordDict`. Return all such sentences.

**Examples**

- `s = 'catsanddog', word_dict = ['cat', 'cats', 'and', 'sand', 'dog']` → `['cats and dog', 'cat sand dog']`
- `s = 'pineapplepenapple', word_dict = ['apple', 'pen', 'applepen', 'pine', 'pineapple']` → `['pine apple pen apple', 'pineapple pen apple', 'pine applepen apple']`
- `s = 'catsandog', word_dict = ['cats', 'dog', 'sand', 'and', 'cat']` → `[]`

In [ ]:
def word_break(s, word_dict):
    """
    Add spaces to s to form sentences where every word is in wordDict. Return
    all such sentences.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(word_break, [(("catsanddog", ["cat", "cats", "and", "sand", "dog"]), ["cats and dog", "cat sand dog"]),
   (("pineapplepenapple", ["apple", "pen", "applepen", "pine", "pineapple"]), ["pine apple pen apple", "pineapple pen apple", "pine applepen apple"]),
   (("catsandog", ["cats", "dog", "sand", "and", "cat"]), [])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try all 2ⁿ⁻¹ ways to place spaces and keep those whose words are all in the dictionary.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

sentences(i) = for every dictionary word starting at i, that word + each sentence of the rest. Memoise on i so each suffix is solved once.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try all 2ⁿ⁻¹ ways to place spaces and keep those whose words are all in the dictionary.

```python
def word_break(s, word_dict):
    words, n, res = set(word_dict), len(s), []
    for mask in range(1 << (n - 1)):
        parts, start = [], 0
        for i in range(n - 1):
            if mask >> i & 1:
                parts.append(s[start:i + 1])
                start = i + 1
        parts.append(s[start:])
        if all(p in words for p in parts):
            res.append(" ".join(parts))
    return res
```

**Cost:** O(n · 2ⁿ) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** sentences(i) = for every dictionary word starting at i, that word + each sentence of the rest. Memoise on i so each suffix is solved once.

```python
def word_break(s, word_dict):
    words = set(word_dict)
    memo = {}

    def go(i):
        if i == len(s):
            return [""]
        if i in memo:
            return memo[i]
        out = []
        for j in range(i + 1, len(s) + 1):
            if s[i:j] in words:
                for rest in go(j):
                    out.append(s[i:j] + (" " + rest if rest else ""))
        memo[i] = out
        return out

    return go(0)
```

**Cost:** O(n · 2ⁿ) output-bound, far less in practice time, O(n · answers) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try all 2ⁿ⁻¹ ways to place spaces and keep those whose words are all in the dictionary. That costs O(n · 2ⁿ) time and O(n) space."

"I can do better: sentences(i) = for every dictionary word starting at i, that word + each sentence of the rest. Memoise on i so each suffix is solved once. That's O(n · 2ⁿ) output-bound, far less in practice time and O(n · answers) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'catsanddog', word_dict = ['cat', 'cats', 'and', 'sand', 'dog']`

1. Sentences for suffix 'dog': ['dog']
2. Sentences for suffix 'sanddog': ['sand dog']
3. Suffix 'dog' already solved → reuse
4. Sentences for suffix 'anddog': ['and dog']
5. Sentences for suffix 'catsanddog': ['cat sand dog', 'cats and dog']

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/backtracking/word-break-ii)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is the base case and why is it needed?</b></summary>

The condition under which the function returns without recursing. Without it (or without progress toward it), calls never stop and the program crashes with a stack overflow.

</details>

<details>
<summary><b>2. Backtracking vs brute force?</b></summary>

Brute force generates every complete candidate and tests it. Backtracking builds candidates incrementally and stops extending a partial candidate as soon as it violates a constraint, skipping whole subtrees.

</details>

<details>
<summary><b>3. Backtracking vs dynamic programming?</b></summary>

Backtracking explores choices and is used to enumerate or find solutions; it's exponential. DP applies when sub-problems overlap and the answer depends only on a small state, so results can be cached — often making it polynomial. Memoised backtracking is DP.

</details>

<details>
<summary><b>4. What is tail recursion?</b></summary>

Recursion where the recursive call is the final operation, so no work remains afterwards. It can be optimised into a loop by compilers that support tail-call optimisation (Python doesn't).

</details>

<details>
<summary><b>5. What is the time complexity of generating all subsets / permutations?</b></summary>

Subsets: 2ⁿ subsets, each up to n long to copy → O(n · 2ⁿ). Permutations: n! of length n → O(n · n!).

</details>

<details>
<summary><b>6. Why copy the path when recording an answer?</b></summary>

The path list is shared and modified throughout the search. Appending the list itself stores a reference that later changes; `path[:]` stores a snapshot.

</details>

<details>
<summary><b>7. How do you avoid duplicate results when the input has duplicates?</b></summary>

Sort the input and, within the same recursion level, skip an element equal to the previous one (`i > start and a[i] == a[i−1]`). For permutations, skip when the previous equal element hasn't been used.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Recursion: a function that calls itself
- [ ] I can explain: The decision tree
- [ ] I can explain: The backtracking template: choose, explore, un-choose
- [ ] I can explain: Permutations
- [ ] I can explain: Pruning and duplicates
- [ ] I can explain: Constraint problems
- [ ] I can explain: From backtracking to memoisation

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Partition to K Equal Sum Subsets
- [ ] [Q17](#q17) Matchsticks to Square
- [ ] [Q18](#q18) N-Queens
- [ ] [Q19](#q19) Sudoku Solver
- [ ] [Q20](#q20) Word Break II

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.